In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

from agents.classifying_agent import ClassifyingAgent
import chromadb
from agents.agent import Agent

In [2]:
DB = "products_vectorstore"
client = chromadb.PersistentClient(path=DB)
collections = client.get_or_create_collection('products')

In [3]:
# classifyingAgent = ClassifyingAgent(collection=collections)


def classifyingAgent(product_description):
    # agent = Agent(name="classifying_agent", description="Categorizes a product based on its description. If the product is difficult to categorize, the agent will return the 3 categories you think it could belong to.", tools=[{"type": "function", "function": classifyingAgent}])
    # response = agent.run(product_description=product_description)
    # return response
    return "baby products, toys, home goods"

categorization_agent = {
    "name": "categorization_agent",
    "description": "Categorizes a product based on its description. If the product is difficult to categorize, the agent will return the 3 categories you think it could belong to.",
    "parameters": {
        "type": "object",
        "properties": {
            "product_description": {
                "type": "string",
                "description": "A description of the product that needs to be categorized."
            },
        },
        "required": ["product_description"],
        "additionalProperties": False
    }
}


# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

# tools = [{"type": "function", "function": price_function}]

In [4]:

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()


OpenAI API Key exists and begins sk-proj-


In [ ]:
# tools = [{"type": "function", "function": categorization_agent}]

tools = [{"type": "function", "function": price_function},{"type": "function", "function": categorization_agent}]

def handle_tool_call(message):
    """
    Actually call the tools associated with this message
    """
    # mapping = {
    #     "categorization_agent": classifyingAgent.classify,
    # }
    mapping = {
        "categorization_agent": classifyingAgent,
    }
    results = []
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = mapping.get(tool_name)
        result = tool(**arguments) if tool else ""
        results.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})
    return results



# def handle_tool_call(message):
#     tool_call = message.tool_calls[0]
#     if tool_call.function.name == "get_ticket_price":
#         arguments = json.loads(tool_call.function.arguments)
#         city = arguments.get('destination_city')
#         price_details = get_ticket_price(city)
#         response = {
#             "role": "tool",
#             "content": price_details,
#             "tool_call_id": tool_call.id
#         }
#     return response





In [ ]:
WELCOME_MESSAGE = "Hi! My name is Peeta, and I'm here to help you register your product and air travel. First, please provide me with a description of your product or where you want to travel to, and I'll take care of the rest!"
system_message ="""
You are a helpful assistant for an e-commerce platform that helps bussinesses register their products and airtravel agent. The customers will provide you with a description of their product. You will then use this information to categorize the product into the correct department.
If the attention of a manager is required, let the business owner know that a human will take care of their case.
"""

# WELCOME_MESSAGE = "Hi! My name is Peeta, and I'm here to help you buy the ticket."

# system_message = """
# You are a helpful assistant for an Airline called FlightAI.
# Give short, courteous answers, no more than 1 sentence.
# Always be accurate. If you don't know the answer, say so.
# """

In [7]:
# I want to reguster a product. It is a baby stroller that can be folded and has a storage basket underneath.

In [ ]:

chatbot = gr.Chatbot(
    type="messages",
    value=[
        {"role": "assistant", "content": WELCOME_MESSAGE}
    ]
)

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print("this is smessage", message)
        response = handle_tool_call(message)
        print("this is response", response)
        messages.append(message)
        # messages.append(message.model_dump(exclude_none=True))
        messages.append(response)
        print("this is messages after appending", messages)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
        print("this is final response", response)
    
    return response.choices[0].message.content

gr.ChatInterface(fn=chat,chatbot=chatbot ,type="messages").launch()

/var/folders/_t/4vpfft894ddd5ymy303fdf2m0000gn/T/ipykernel_17396/3783983500.py:1: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


this is smessage ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_ludmTnQhDHjzVJUuAxD6CfJ6', function=Function(arguments='{"product_description":"cat food"}', name='categorization_agent'), type='function')])
this is response [{'role': 'tool', 'content': 'baby products, toys, home goods', 'tool_call_id': 'call_ludmTnQhDHjzVJUuAxD6CfJ6'}]
this is messages after appending [{'role': 'system', 'content': '\nYou are a helpful assistant for an e-commerce platform that helps bussinesses register their products. The customers will provide you with a description of their product. You will then use this information to categorize the product into the correct department.\nIf the attention of a manager is required, let the business owner know that a human will take care of their case.\n'}, {'role': 'assistant', 'content': "Hi! My name is Peeta, and I'm here to help you regist

Traceback (most recent call last):
  File "/Users/arumlee/projectLLMAgent/LLM_AgenticAI/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/arumlee/projectLLMAgent/LLM_AgenticAI/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/arumlee/projectLLMAgent/LLM_AgenticAI/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/arumlee/projectLLMAgent/LLM_AgenticAI/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1696, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/arumlee/projectLLMAgent/LLM_AgenticAI